In [1]:
import pandas as pd

In [2]:
import tarfile
import io

# Extract and read from tar.gz archive
archive_path = 'data/hwu.tar.gz'

def read_from_tar(archive, filename):
    with tarfile.open(archive, 'r:gz') as tar:
        member = tar.getmember(filename)
        f = tar.extractfile(member)
        return pd.read_csv(io.TextIOWrapper(f, encoding='utf-8'), sep=',', skiprows=1, header=None, names=['text', 'intent'])

df_train = read_from_tar(archive_path, 'hwu/train.csv')
df_val = read_from_tar(archive_path, 'hwu/val.csv')
df_test = read_from_tar(archive_path, 'hwu/test.csv')


In [12]:
print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)
df_train.head()

Train shape: (8954, 2)
Validation shape: (1076, 2)
Test shape: (1076, 2)


,text,intent
0,what alarms do i have set right now,alarm_query
1,checkout today alarm of meeting,alarm_query
2,report alarm settings,alarm_query
3,see see for me the alarms that you have set to...,alarm_query
4,is there an alarm for ten am,alarm_query


In [3]:
from sklearn.preprocessing import LabelEncoder

# Encode intent labels to numeric values
label_encoder = LabelEncoder()

# Fit on all unique intents from train set
label_encoder.fit(df_train['intent'])

# Transform all splits
df_train['label'] = label_encoder.transform(df_train['intent'])
df_val['label'] = label_encoder.transform(df_val['intent'])
df_test['label'] = label_encoder.transform(df_test['intent'])

print(f"Number of classes: {len(label_encoder.classes_)}")
print(f"Sample classes: {label_encoder.classes_[:5]}")
print(f"\nLabel distribution in train:")
print(df_train['label'].value_counts().head())

Number of classes: 64
Sample classes: ['alarm_query' 'alarm_remove' 'alarm_set' 'audio_volume_down'
 'audio_volume_mute']

Label distribution in train:
label
2     159
55    159
48    159
0     158
24    158
Name: count, dtype: int64


In [4]:
# Task 1: Baseline - TF-IDF + Logistic Regression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score

# 1. Create pipeline with TfidfVectorizer and LogisticRegression
tfidf_lr_pipeline = make_pipeline(
    TfidfVectorizer(max_features=5000),
    LogisticRegression(max_iter=1000, random_state=42)
)

# 2. Train pipeline on train set
print("Training TF-IDF + Logistic Regression baseline...")
tfidf_lr_pipeline.fit(df_train['text'], df_train['label'])

# 3. Evaluate on test set
y_pred_baseline = tfidf_lr_pipeline.predict(df_test['text'])
baseline_accuracy = accuracy_score(df_test['label'], y_pred_baseline)

print(f"\nBaseline Accuracy: {baseline_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(df_test['label'], y_pred_baseline, target_names=label_encoder.classes_, zero_division=0))

Training TF-IDF + Logistic Regression baseline...

Baseline Accuracy: 0.8355

Classification Report:
                          precision    recall  f1-score   support

             alarm_query       0.90      0.95      0.92        19
            alarm_remove       1.00      0.73      0.84        11
               alarm_set       0.77      0.89      0.83        19
       audio_volume_down       1.00      0.75      0.86         8
       audio_volume_mute       0.92      0.80      0.86        15
         audio_volume_up       0.93      1.00      0.96        13
          calendar_query       0.45      0.53      0.49        19
         calendar_remove       0.89      0.89      0.89        19
            calendar_set       0.87      0.68      0.76        19
          cooking_recipe       0.59      0.68      0.63        19
        datetime_convert       0.67      0.75      0.71         8
          datetime_query       0.74      0.89      0.81        19
        email_addcontact       0.78     

In [5]:
# Task 2: Word2Vec (Average) + Dense Layer
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.models import Word2Vec

# Train Word2Vec model
print("Training Word2Vec model...")
sentences = [text.split() for text in df_train['text']]
w2v_model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=1, workers=4, seed=42)
print(f"Word2Vec vocabulary size: {len(w2v_model.wv)}")

# Function to create average embedding vectors using Word2Vec
def sentence_to_avg_vector(text, model):
    words = text.split()
    vectors = [model.wv[word] for word in words if word in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

# Create average vectors
X_train_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_train['text']])
X_val_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_val['text']])
X_test_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_test['text']])

# Convert labels to categorical
num_classes = len(label_encoder.classes_)
y_train_cat = to_categorical(df_train['label'], num_classes)
y_val_cat = to_categorical(df_val['label'], num_classes)
y_test_cat = to_categorical(df_test['label'], num_classes)

print(f"X_train_avg shape: {X_train_avg.shape}")
print(f"X_val_avg shape: {X_val_avg.shape}")
print(f"X_test_avg shape: {X_test_avg.shape}")

# Build Dense model
print("\nBuilding Dense model...")
dense_model = Sequential([
    Dense(128, activation='relu', input_shape=(100,)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

dense_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(dense_model.summary())

print("\nTraining Dense model...")
history = dense_model.fit(
    X_train_avg, y_train_cat,
    validation_data=(X_val_avg, y_val_cat),
    epochs=20,
    batch_size=32,
    verbose=1
)

# Evaluate on test set
test_loss, test_accuracy = dense_model.evaluate(X_test_avg, y_test_cat, verbose=0)
print(f"\nAverage Embedding + Dense Model Test Accuracy: {test_accuracy:.4f}")


Training Word2Vec model...
Word2Vec vocabulary size: 4467
X_train_avg shape: (8954, 100)
X_val_avg shape: (1076, 100)
X_test_avg shape: (1076, 100)

Building Dense model...
X_train_avg shape: (8954, 100)
X_val_avg shape: (1076, 100)
X_test_avg shape: (1076, 100)

Building Dense model...


C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        12,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,344 (99.00 KB)

 Trainable params: 25,344 (99.00 KB)

 Non-trainable params: 0 (0.00 B)

None

Training Dense model...
Epoch 1/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0163 - loss: 4.1526 - val_accuracy: 0.0511 - val_loss: 4.1276
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0163 - loss: 4.1526 - val_accuracy: 0.0511 - val_loss: 4.1276
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0249 - loss: 4.1210 - val_accuracy: 0.0381 - val_loss: 4.0728
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0249 - loss: 4.1210 - val_accuracy: 0.0381 - val_loss: 4.0728
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0398 - loss: 4.0097 - val_accuracy: 0.0558 - val_loss: 3.8614
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0398 - loss: 4.0097 - val_accuracy: 0.0558 - val_loss: 3.8614
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0548 - loss: 3.8460 - val_accuracy: 0.0734 - val_loss: 3.7322
Epoch 5/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0548 - l

In [6]:
# Task 3: Advanced Model (Pre-trained Embedding + LSTM)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM
from tensorflow.keras.callbacks import EarlyStopping

# 1. Tiền xử lý cho mô hình chuỗi
# a. Tokenizer: Tạo vocab và chuyển text thành chuỗi chỉ số
tokenizer = Tokenizer(num_words=len(w2v_model.wv), oov_token="<UNK>")
tokenizer.fit_on_texts(df_train['text'])
train_sequences = tokenizer.texts_to_sequences(df_train['text'])
val_sequences = tokenizer.texts_to_sequences(df_val['text'])
test_sequences = tokenizer.texts_to_sequences(df_test['text'])

# b. Padding: Đảm bảo các chuỗi có cùng độ dài
max_len = 50
X_train_pad = pad_sequences(train_sequences, maxlen=max_len, padding='post')
X_val_pad = pad_sequences(val_sequences, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(test_sequences, maxlen=max_len, padding='post')

print(f"X_train_pad shape: {X_train_pad.shape}")
print(f"X_val_pad shape: {X_val_pad.shape}")
print(f"X_test_pad shape: {X_test_pad.shape}")

# 2. Tạo ma trận trọng số cho Embedding Layer từ Word2Vec
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = w2v_model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

# Load embeddings from Word2Vec
for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

print(f"Embedding matrix shape: {embedding_matrix.shape}")

# 3. Xây dựng mô hình Sequential với LSTM
lstm_model_pretrained = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],  # Khởi tạo trọng số
        input_length=max_len,
        trainable=False  # Đóng băng lớp Embedding
    ),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(num_classes, activation='softmax')
])

lstm_model_pretrained.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(lstm_model_pretrained.summary())

# 4. Compile, huấn luyện (sử dụng EarlyStopping) và đánh giá
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

print("\nTraining LSTM with frozen pre-trained embeddings...")
history_pretrained = lstm_model_pretrained.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=30,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluate
test_loss_pretrained, test_acc_pretrained = lstm_model_pretrained.evaluate(X_test_pad, y_test_cat, verbose=0)
print(f"\nPre-trained Frozen Embedding + LSTM Test Accuracy: {test_acc_pretrained:.4f}")


X_train_pad shape: (8954, 50)
X_val_pad shape: (1076, 50)
X_test_pad shape: (1076, 50)
Embedding matrix shape: (4265, 100)


C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │       426,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 426,500 (1.63 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 426,500 (1.63 MB)

None

Training LSTM with frozen pre-trained embeddings...
Epoch 1/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - accuracy: 0.0166 - loss: 4.1435 - val_accuracy: 0.0167 - val_loss: 4.1313
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - accuracy: 0.0166 - loss: 4.1435 - val_accuracy: 0.0167 - val_loss: 4.1313
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 50ms/step - accuracy: 0.0305 - loss: 4.0747 - val_accuracy: 0.0418 - val_loss: 3.9438
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 50ms/step - accuracy: 0.0305 - loss: 4.0747 - val_accuracy: 0.0418 - val_loss: 3.9438
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 48ms/step - accuracy: 0.0380 - loss: 3.9903 - val_accuracy: 0.0493 - val_loss: 3.8751
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 7s 48ms/step - accuracy: 0.0380 - loss: 3.9903 - val_accuracy: 0.0493 - val_loss: 3.8751
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step - accuracy: 0.0471 - loss: 3.9206 - val_accuracy: 0.0623 - val_loss: 3.8365
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━

In [7]:
# Task 4: Advanced Model (Embedding học từ đầu + LSTM)
# Dữ liệu đã được tiền xử lý (tokenized, padded) từ nhiệm vụ 3

# 1. Xây dựng mô hình
lstm_model_scratch = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=100,  # Chọn một chiều embedding, ví dụ 100
        input_length=max_len
        # Không có weights, trainable=True (mặc định)
    ),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(num_classes, activation='softmax')
])

# 2. Compile, huấn luyện và đánh giá mô hình
lstm_model_scratch.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(lstm_model_scratch.summary())

# Train with EarlyStopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

print("\nTraining LSTM with trainable embeddings from scratch...")
history_scratch = lstm_model_scratch.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=30,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluate
test_loss_scratch, test_acc_scratch = lstm_model_scratch.evaluate(X_test_pad, y_test_cat, verbose=0)
print(f"\nTrainable Embedding + LSTM Test Accuracy: {test_acc_scratch:.4f}")

# Final comparison of all models
print("\n" + "="*70)
print("FINAL MODEL COMPARISON:")
print("="*70)
print(f"1. TF-IDF + Logistic Regression:        {baseline_accuracy:.4f}")
print(f"2. Average Embedding + Dense:           {test_accuracy:.4f}")
print(f"3. Pre-trained Frozen Embedding + LSTM: {test_acc_pretrained:.4f}")
print(f"4. Trainable Embedding + LSTM:          {test_acc_scratch:.4f}")
print("="*70)

best_acc = max(baseline_accuracy, test_accuracy, test_acc_pretrained, test_acc_scratch)
if best_acc == test_acc_scratch:
    print("\nBest Model: LSTM with Trainable Embeddings (Task 4)")
elif best_acc == test_acc_pretrained:
    print("\nBest Model: LSTM with Frozen Pre-trained Embeddings (Task 3)")
elif best_acc == baseline_accuracy:
    print("\nBest Model: TF-IDF + Logistic Regression (Baseline)")
else:
    print("\nBest Model: Average Embedding + Dense (Task 2)")


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None

Training LSTM with trainable embeddings from scratch...
Epoch 1/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 57ms/step - accuracy: 0.0170 - loss: 4.1452 - val_accuracy: 0.0177 - val_loss: 4.1286
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 57ms/step - accuracy: 0.0170 - loss: 4.1452 - val_accuracy: 0.0177 - val_loss: 4.1286
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 0.0136 - loss: 4.1358 - val_accuracy: 0.0177 - val_loss: 4.1269
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 0.0136 - loss: 4.1358 - val_accuracy: 0.0177 - val_loss: 4.1269
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 57ms/step - accuracy: 0.0156 - loss: 4.1336 - val_accuracy: 0.0177 - val_loss: 4.1258
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 57ms/step - accuracy: 0.0156 - loss: 4.1336 - val_accuracy: 0.0177 - val_loss: 4.1258
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 58ms/step - accuracy: 0.0164 - loss: 4.1334 - val_accuracy: 0.0177 - val_loss: 4.1264
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━

In [8]:
# Display final results
print("="*70)
print("FINAL MODEL COMPARISON:")
print("="*70)
print(f"1. TF-IDF + Logistic Regression:        {baseline_accuracy:.4f}")
print(f"2. Average Embedding + Dense:           {test_accuracy:.4f}")
print(f"3. Pre-trained Frozen Embedding + LSTM: {test_acc_pretrained:.4f}")
print(f"4. Trainable Embedding + LSTM:          {test_acc_scratch:.4f}")
print("="*70)

best_acc = max(baseline_accuracy, test_accuracy, test_acc_pretrained, test_acc_scratch)
if best_acc == test_acc_scratch:
    print("\nBest Model: LSTM with Trainable Embeddings (Task 4)")
elif best_acc == test_acc_pretrained:
    print("\nBest Model: LSTM with Frozen Pre-trained Embeddings (Task 3)")
elif best_acc == baseline_accuracy:
    print("\nBest Model: TF-IDF + Logistic Regression (Baseline)")
else:
    print("\nBest Model: Average Embedding + Dense (Task 2)")


FINAL MODEL COMPARISON:
1. TF-IDF + Logistic Regression:        0.8355
2. Average Embedding + Dense:           0.2017
3. Pre-trained Frozen Embedding + LSTM: 0.0985
4. Trainable Embedding + LSTM:          0.0177

Best Model: TF-IDF + Logistic Regression (Baseline)


In [9]:
# Task 5: Đánh giá, So sánh và Phân tích

# 1. So sánh định lượng: Tính F1-score (Macro) cho tất cả models
from sklearn.metrics import f1_score

print("="*70)
print("1. SO SÁNH ĐỊNH LƯỢNG")
print("="*70)

# Get predictions from all models
y_true = df_test['label']

# TF-IDF + LR
y_pred_tfidf = tfidf_lr_pipeline.predict(df_test['text'])
f1_tfidf = f1_score(y_true, y_pred_tfidf, average='macro')

# Word2Vec Avg + Dense
y_pred_dense = dense_model.predict(X_test_avg, verbose=0)
y_pred_dense_labels = np.argmax(y_pred_dense, axis=1)
f1_dense = f1_score(y_true, y_pred_dense_labels, average='macro')

# Pre-trained LSTM
y_pred_pretrained = lstm_model_pretrained.predict(X_test_pad, verbose=0)
y_pred_pretrained_labels = np.argmax(y_pred_pretrained, axis=1)
f1_pretrained = f1_score(y_true, y_pred_pretrained_labels, average='macro')

# Trainable LSTM
y_pred_scratch = lstm_model_scratch.predict(X_test_pad, verbose=0)
y_pred_scratch_labels = np.argmax(y_pred_scratch, axis=1)
f1_scratch = f1_score(y_true, y_pred_scratch_labels, average='macro')

# Create comparison table
print("\nBảng so sánh kết quả:")
print("-"*70)
print(f"{'Pipeline':<40} {'F1-score (Macro)':<20} {'Test Loss':<10}")
print("-"*70)
print(f"{'TF-IDF + Logistic Regression':<40} {f1_tfidf:<20.4f} {'N/A':<10}")
print(f"{'Word2Vec (Avg) + Dense':<40} {f1_dense:<20.4f} {test_loss:<10.4f}")
print(f"{'Embedding (Pre-trained) + LSTM':<40} {f1_pretrained:<20.4f} {test_loss_pretrained:<10.4f}")
print(f"{'Embedding (Scratch) + LSTM':<40} {f1_scratch:<20.4f} {test_loss_scratch:<10.4f}")
print("-"*70)

print(f"\nKết luận:")
print(f"- Model tốt nhất về F1-score: {'TF-IDF + LR' if f1_tfidf == max(f1_tfidf, f1_dense, f1_pretrained, f1_scratch) else 'Dense' if f1_dense == max(f1_tfidf, f1_dense, f1_pretrained, f1_scratch) else 'Pre-trained LSTM' if f1_pretrained == max(f1_tfidf, f1_dense, f1_pretrained, f1_scratch) else 'Trainable LSTM'}")
print(f"- F1-score macro quan trọng vì dataset có 64 classes cân bằng")
print(f"- TF-IDF baseline vượt trội cho bài toán intent classification ngắn")


1. SO SÁNH ĐỊNH LƯỢNG

Bảng so sánh kết quả:
----------------------------------------------------------------------
Pipeline                                 F1-score (Macro)     Test Loss 
----------------------------------------------------------------------
TF-IDF + Logistic Regression             0.8353               N/A       
Word2Vec (Avg) + Dense                   0.1413               3.0267    
Embedding (Pre-trained) + LSTM           0.0470               3.4784    
Embedding (Scratch) + LSTM               0.0005               4.1234    
----------------------------------------------------------------------

Kết luận:
- Model tốt nhất về F1-score: TF-IDF + LR
- F1-score macro quan trọng vì dataset có 64 classes cân bằng
- TF-IDF baseline vượt trội cho bài toán intent classification ngắn

Bảng so sánh kết quả:
----------------------------------------------------------------------
Pipeline                                 F1-score (Macro)     Test Loss 
---------------------------

In [10]:
# 2. Phân tích định tính: Kiểm tra câu phức tạp

print("\n" + "="*70)
print("2. PHÂN TÍCH ĐỊNH TÍNH - Kiểm tra câu phức tạp")
print("="*70)

# Test sentences (find similar ones in our dataset or create new ones)
test_sentences = [
    "can you remind me to not call my mom",
    "is it going to be sunny or rainy tomorrow", 
    "find a flight from new york to london but not through paris",
    "do not play that song",
    "what is the weather but not for today"
]

print("\nChú ý: Các câu test phức tạp có thể không có trong dataset HWU.")
print("Dataset HWU chủ yếu chứa câu lệnh đơn giản cho assistant.\n")

# Find some complex sentences from actual test set
print("Tìm các câu phức tạp trong test set thực tế:")
print("-"*70)

# Get some examples with negation or complex structure from test set
complex_indices = []
for idx, text in enumerate(df_test['text'].values):
    if any(word in text.lower() for word in ['not', 'no', 'but', 'or', 'without', 'except']):
        complex_indices.append(idx)
    if len(complex_indices) >= 5:
        break

if len(complex_indices) == 0:
    # If no complex sentences, just take random samples
    complex_indices = [10, 25, 50, 75, 100]

for idx in complex_indices:
    text = df_test['text'].iloc[idx]
    true_label = label_encoder.classes_[df_test['label'].iloc[idx]]
    
    print(f"\nCâu test: \"{text}\"")
    print(f"Nhãn thật: {true_label}")
    print("-"*70)
    
    # TF-IDF prediction
    pred_tfidf = tfidf_lr_pipeline.predict([text])[0]
    pred_tfidf_label = label_encoder.classes_[pred_tfidf]
    
    # Dense prediction
    text_vector = sentence_to_avg_vector(text, w2v_model).reshape(1, -1)
    pred_dense = dense_model.predict(text_vector, verbose=0)
    pred_dense_label = label_encoder.classes_[np.argmax(pred_dense)]
    
    # Pre-trained LSTM prediction
    text_seq = tokenizer.texts_to_sequences([text])
    text_pad = pad_sequences(text_seq, maxlen=max_len, padding='post')
    pred_pretrained = lstm_model_pretrained.predict(text_pad, verbose=0)
    pred_pretrained_label = label_encoder.classes_[np.argmax(pred_pretrained)]
    
    # Trainable LSTM prediction
    pred_scratch = lstm_model_scratch.predict(text_pad, verbose=0)
    pred_scratch_label = label_encoder.classes_[np.argmax(pred_scratch)]
    
    # Display predictions
    correct_symbol = "✓" if pred_tfidf_label == true_label else "✗"
    print(f"TF-IDF + LR:           {pred_tfidf_label:<30} {correct_symbol}")
    
    correct_symbol = "✓" if pred_dense_label == true_label else "✗"
    print(f"Word2Vec Avg + Dense:  {pred_dense_label:<30} {correct_symbol}")
    
    correct_symbol = "✓" if pred_pretrained_label == true_label else "✗"
    print(f"Pre-trained LSTM:      {pred_pretrained_label:<30} {correct_symbol}")
    
    correct_symbol = "✓" if pred_scratch_label == true_label else "✗"
    print(f"Trainable LSTM:        {pred_scratch_label:<30} {correct_symbol}")
    print()



2. PHÂN TÍCH ĐỊNH TÍNH - Kiểm tra câu phức tạp

Chú ý: Các câu test phức tạp có thể không có trong dataset HWU.
Dataset HWU chủ yếu chứa câu lệnh đơn giản cho assistant.

Tìm các câu phức tạp trong test set thực tế:
----------------------------------------------------------------------

Câu test: "what alarms are set for today"
Nhãn thật: alarm_query
----------------------------------------------------------------------
TF-IDF + LR:           alarm_query                    ✓
Word2Vec Avg + Dense:  alarm_query                    ✓
Pre-trained LSTM:      alarm_set                      ✗
Trainable LSTM:        social_post                    ✗


Câu test: "please see see for me the alarms that you have set sunday morning"
Nhãn thật: alarm_query
----------------------------------------------------------------------
TF-IDF + LR:           alarm_query                    ✓
Word2Vec Avg + Dense:  audio_volume_mute              ✗
Pre-trained LSTM:      qa_currency                    ✗
Trainable

In [11]:
# 3. Nhận xét và phân tích

print("="*70)
print("3. NHẬN XÉT VÀ PHÂN TÍCH")
print("="*70)

print("""
KẾT QUẢ TỔNG QUAN:
- TF-IDF + Logistic Regression đạt hiệu suất tốt nhất (~83.5%)
- Các mô hình deep learning (Dense, LSTM) có hiệu suất thấp hơn nhiều

TẠI SAO TF-IDF THẮNG?
1. Đặc điểm bài toán:
   - Intent classification với câu ngắn (thường 5-10 từ)
   - 64 classes với vocabulary hạn chế trong domain assistant
   - Các intent phân biệt chủ yếu bằng keywords quan trọng
   - Ví dụ: "weather" → weather_query, "alarm" → alarm_set
   
2. Ưu điểm của TF-IDF:
   - Bắt keywords quan trọng tốt với tf-idf weighting
   - Linear model đủ cho decision boundary đơn giản
   - Ít tham số → ít overfitting với dataset nhỏ (~9K samples)
   
3. Hạn chế của Deep Learning models:
   - Cần nhiều data hơn để học được representation tốt
   - LSTM models bị underfitting nghiêm trọng (1-10% accuracy)
   - Word2Vec training trên corpus nhỏ không cho embeddings tốt
   - Overfitting do quá nhiều parameters cho dataset size này

KHẢ NĂNG XỬ LÝ CHUỖI CỦA LSTM:
- Về lý thuyết, LSTM nên tốt hơn cho:
  + Câu có phủ định: "do NOT play that song"
  + Câu phức: "remind me to call mom at 5pm tomorrow"
  + Context dependencies: word order matters
  
- Trong thực tế với dataset này:
  + LSTM không học được do thiếu data và embeddings yếu
  + Intent classification trong HWU dataset đơn giản, dựa keywords
  + Không cần sequential modeling phức tạp
  
ĐỀ XUẤT CẢI THIỆN LSTM:
1. Sử dụng pre-trained embeddings tốt hơn (GloVe, FastText, BERT)
2. Data augmentation để tăng training samples
3. Transfer learning từ models đã train trên corpus lớn
4. Fine-tune learning rate, thử optimizer khác (SGD with momentum)
5. Thử architecture đơn giản hơn (GRU thay vì LSTM)
6. Bidirectional LSTM để capture context tốt hơn

KẾT LUẬN:
- Baseline TF-IDF + LR là lựa chọn tốt nhất cho bài toán này
- Deep learning cần nhiều data và engineering hơn mới hiệu quả
- Simple models often win with small, structured datasets
- "No Free Lunch" theorem: không có model nào tốt cho mọi bài toán
""")


3. NHẬN XÉT VÀ PHÂN TÍCH

KẾT QUẢ TỔNG QUAN:
- TF-IDF + Logistic Regression đạt hiệu suất tốt nhất (~83.5%)
- Các mô hình deep learning (Dense, LSTM) có hiệu suất thấp hơn nhiều

TẠI SAO TF-IDF THẮNG?
1. Đặc điểm bài toán:
   - Intent classification với câu ngắn (thường 5-10 từ)
   - 64 classes với vocabulary hạn chế trong domain assistant
   - Các intent phân biệt chủ yếu bằng keywords quan trọng
   - Ví dụ: "weather" → weather_query, "alarm" → alarm_set

2. Ưu điểm của TF-IDF:
   - Bắt keywords quan trọng tốt với tf-idf weighting
   - Linear model đủ cho decision boundary đơn giản
   - Ít tham số → ít overfitting với dataset nhỏ (~9K samples)

3. Hạn chế của Deep Learning models:
   - Cần nhiều data hơn để học được representation tốt
   - LSTM models bị underfitting nghiêm trọng (1-10% accuracy)
   - Word2Vec training trên corpus nhỏ không cho embeddings tốt
   - Overfitting do quá nhiều parameters cho dataset size này

KHẢ NĂNG XỬ LÝ CHUỖI CỦA LSTM:
- Về lý thuyết, LSTM nên tốt hơn 